# AllSchoolsColleges — Python Analytics

## Objective
Analyze the school acquisition and enrollment funnel from lead generation
through applications and enrollments, with a focus on marketing efficiency,
regional performance, and revenue.

## Tools
- Python
- Pandas
- NumPy
- BigQuery
- Matplotlib

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "project-53983553-3770-4fa0-a04"

client = bigquery.Client(project=PROJECT_ID)

print("BigQuery connected successfully!")

BigQuery connected successfully!


In [2]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "project-53983553-3770-4fa0-a04"
DATASET = "allschoolscolleges"

client = bigquery.Client(project=PROJECT_ID)

print("BigQuery client connected.")

BigQuery client connected.


## 1. Data Loading & Quality Assessment

The analysis uses four core business tables from BigQuery:

- `dim_school`
- `fact_leads`
- `fact_applications`
- `fact_enrollments`

A separate PPC fact table, `fact_ppc`, is used for campaign performance analysis.

Data quality checks included:
- Missing-value analysis
- Duplicate detection
- Primary-key uniqueness
- Categorical distribution checks
- Detection and removal of an accidental header row

In [3]:
query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.dim_school`
"""

df_school = client.query(query).to_dataframe()

print("Rows:", len(df_school))
print("Columns:", len(df_school.columns))

df_school.head()

Rows: 101
Columns: 7


,school_id,school_name,city,state,region,school_type,boarding_type
0,SCH010,Delhi Public School,Patna,Bihar,East,Private,Day
1,SCH043,JG International School,Ahmedabad,Gujarat,West,Private,Day
2,SCH045,Navrachana International School,Vadodara,Gujarat,West,Private,Day
3,SCH048,Bhavkunj School,Kadi,Gujarat,West,Private,Day
4,SCH049,PD Savani Cambridge International School,Surat,Gujarat,West,Private,Day


In [4]:
df_school.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   school_id      101 non-null    object
 1   school_name    101 non-null    object
 2   city           101 non-null    object
 3   state          101 non-null    object
 4   region         101 non-null    object
 5   school_type    101 non-null    object
 6   boarding_type  101 non-null    object
dtypes: object(7)
memory usage: 5.7+ KB


In [5]:
df_school.head()

,school_id,school_name,city,state,region,school_type,boarding_type
0,SCH010,Delhi Public School,Patna,Bihar,East,Private,Day
1,SCH043,JG International School,Ahmedabad,Gujarat,West,Private,Day
2,SCH045,Navrachana International School,Vadodara,Gujarat,West,Private,Day
3,SCH048,Bhavkunj School,Kadi,Gujarat,West,Private,Day
4,SCH049,PD Savani Cambridge International School,Surat,Gujarat,West,Private,Day


In [6]:
# Basic data quality check

print("Shape:", df_school.shape)
print("\nMissing values:")
print(df_school.isnull().sum())

print("\nDuplicate rows:", df_school.duplicated().sum())

Shape: (101, 7)

Missing values:
school_id        0
school_name      0
city             0
state            0
region           0
school_type      0
boarding_type    0
dtype: int64

Duplicate rows: 0


In [7]:
df_school.describe(include="all").T

,count,unique,top,freq
school_id,101,101,SCH010,1
school_name,101,101,Delhi Public School,1
city,101,34,Dehradun,23
state,101,15,Uttarakhand,28
region,101,6,North,47
school_type,101,2,Private,100
boarding_type,101,3,Residential,78


## 1. Data Loading & Quality Assessment

The analysis uses four core business tables from BigQuery:

- `dim_school`
- `fact_leads`
- `fact_applications`
- `fact_enrollments`

A separate PPC fact table, `fact_ppc`, is used for campaign performance analysis.

Data quality checks included:
- Missing-value analysis
- Duplicate detection
- Primary-key uniqueness
- Categorical distribution checks
- Detection and removal of an accidental header row

In [8]:
# School distribution by state

state_distribution = (
    df_school
    .groupby("state")
    .size()
    .reset_index(name="school_count")
    .sort_values("school_count", ascending=False)
)

state_distribution

,state,school_count
12,Uttarakhand,28
8,Maharashtra,15
2,Gujarat,12
7,Madhya Pradesh,8
3,Haryana,6
6,Karnataka,5
4,Himachal Pradesh,5
10,Telangana,4
9,Rajasthan,4
0,Bihar,3


In [9]:
# School distribution by region

region_distribution = (
    df_school
    .groupby("region")
    .size()
    .reset_index(name="school_count")
    .sort_values("school_count", ascending=False)
)

region_distribution

,region,school_count
2,North,47
4,West,27
3,South,9
1,East,9
0,Central,8
5,region,1


In [10]:
df_school[
    (df_school["state"] == "state") |
    (df_school["region"] == "region")
]

,school_id,school_name,city,state,region,school_type,boarding_type
100,school_id,school_name,city,state,region,school_type,boarding_type


In [11]:
df_school_clean = df_school[
    df_school["school_id"] != "school_id"
].copy()

print("Original rows:", len(df_school))
print("Clean rows:", len(df_school_clean))
print("Rows removed:", len(df_school) - len(df_school_clean))

Original rows: 101
Clean rows: 100
Rows removed: 1


In [12]:
df_school_clean[df_school_clean["school_id"] == "school_id"]

,school_id,school_name,city,state,region,school_type,boarding_type


In [13]:
print("Shape:", df_school_clean.shape)
print("Duplicates:", df_school_clean.duplicated().sum())
print("\nMissing values:")
print(df_school_clean.isnull().sum())

Shape: (100, 7)
Duplicates: 0

Missing values:
school_id        0
school_name      0
city             0
state            0
region           0
school_type      0
boarding_type    0
dtype: int64


In [14]:
query_leads = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fact_leads`
"""

df_leads = client.query(query_leads).to_dataframe()

print("Shape:", df_leads.shape)
df_leads.head()

Shape: (75000, 9)


,lead_id,student_id,school_id,lead_date,lead_source,traffic_type,target_type,campaign_id,status
0,LED0000003,STU013285,SCH075,2022-01-03 00:00:00+00:00,Organic,Organic Search,School,None,Contacted
1,LED0000017,STU010438,SCH081,2023-10-25 00:00:00+00:00,Organic,Organic Search,School,None,Contacted
2,LED0000036,STU009228,SCH014,2021-08-30 00:00:00+00:00,Organic,Organic Search,School,None,Contacted
3,LED0000037,STU020890,SCH077,2022-08-30 00:00:00+00:00,Organic,Organic Search,Location,None,Contacted
4,LED0000038,STU016763,SCH036,2025-04-06 00:00:00+00:00,Organic,Organic Search,Location,None,Contacted


In [15]:
df_leads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   lead_id       75000 non-null  object             
 1   student_id    75000 non-null  object             
 2   school_id     75000 non-null  object             
 3   lead_date     75000 non-null  datetime64[us, UTC]
 4   lead_source   75000 non-null  object             
 5   traffic_type  75000 non-null  object             
 6   target_type   75000 non-null  object             
 7   campaign_id   31531 non-null  object             
 8   status        75000 non-null  object             
dtypes: datetime64[us, UTC](1), object(8)
memory usage: 5.1+ MB


In [16]:
print("Duplicate rows:", df_leads.duplicated().sum())

print("\nDuplicate Lead IDs:", df_leads["lead_id"].duplicated().sum())

print("\nMissing values:")
print(df_leads.isna().sum())

print("\nLead status:")
print(df_leads["status"].value_counts())

print("\nLead source:")
print(df_leads["lead_source"].value_counts())

Duplicate rows: 0

Duplicate Lead IDs: 0

Missing values:
lead_id             0
student_id          0
school_id           0
lead_date           0
lead_source         0
traffic_type        0
target_type         0
campaign_id     43469
status              0
dtype: int64

Lead status:
status
Contacted    17326
Lost         17146
Converted    16337
Qualified    15062
New           9129
Name: count, dtype: int64

Lead source:
lead_source
Organic    43469
PPC        31531
Name: count, dtype: int64


In [17]:
df_leads_clean = df_leads.copy()

print("Shape:", df_leads_clean.shape)
print("Duplicates:", df_leads_clean.duplicated().sum())
print("Missing campaign IDs:", df_leads_clean["campaign_id"].isna().sum())

Shape: (75000, 9)
Duplicates: 0
Missing campaign IDs: 43469


In [18]:
query_applications = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fact_applications`
"""

df_applications = client.query(query_applications).to_dataframe()

print("Shape:", df_applications.shape)
df_applications.head()

Shape: (20000, 6)


,application_id,student_id,school_id,application_date,grade,status
0,APP002656,STU024974,SCH001,2024-07-04 00:00:00+00:00,Grade 10,Accepted
1,APP005029,STU006726,SCH001,2021-11-11 00:00:00+00:00,Grade 10,Accepted
2,APP008665,STU004437,SCH001,2021-06-04 00:00:00+00:00,Grade 10,Accepted
3,APP008783,STU011119,SCH001,2026-03-07 00:00:00+00:00,Grade 10,Accepted
4,APP010109,STU008280,SCH001,2024-01-22 00:00:00+00:00,Grade 10,Accepted


In [19]:
df_applications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   application_id    20000 non-null  object             
 1   student_id        20000 non-null  object             
 2   school_id         20000 non-null  object             
 3   application_date  20000 non-null  datetime64[us, UTC]
 4   grade             20000 non-null  object             
 5   status            20000 non-null  object             
dtypes: datetime64[us, UTC](1), object(5)
memory usage: 937.6+ KB


In [20]:
print("Duplicate rows:", df_applications.duplicated().sum())

print("Duplicate Application IDs:",
      df_applications["application_id"].duplicated().sum())

print("\nApplication status:")
print(df_applications["status"].value_counts())

print("\nGrade distribution:")
print(df_applications["grade"].value_counts())

Duplicate rows: 0
Duplicate Application IDs: 0

Application status:
status
Applied         7597
Accepted        6332
Under Review    3693
Rejected        2378
Name: count, dtype: int64

Grade distribution:
grade
Grade 9     3458
Grade 10    3346
Grade 6     3325
Grade 8     3307
Grade 11    3286
Grade 7     3278
Name: count, dtype: int64


In [21]:
df_applications_clean = df_applications.copy()

print("Shape:", df_applications_clean.shape)
print("Duplicates:", df_applications_clean.duplicated().sum())

Shape: (20000, 6)
Duplicates: 0


In [22]:
query_enrollments = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.fact_enrollments`
"""

df_enrollments = client.query(query_enrollments).to_dataframe()

print("Shape:", df_enrollments.shape)
df_enrollments.head()

Shape: (6332, 6)


,enrollment_id,student_id,school_id,enrolled_at,status,fee_amount
0,ENR000014,STU000460,SCH001,2025-08-05 00:00:00+00:00,Enrolled,100558
1,ENR000015,STU023021,SCH001,2023-08-09 00:00:00+00:00,Enrolled,236966
2,ENR000016,STU015169,SCH001,2026-02-02 00:00:00+00:00,Enrolled,237865
3,ENR000145,STU018744,SCH001,2021-12-26 00:00:00+00:00,Enrolled,222825
4,ENR000174,STU011480,SCH001,2024-10-23 00:00:00+00:00,Enrolled,107310


In [23]:
df_enrollments.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6332 entries, 0 to 6331
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   enrollment_id  6332 non-null   object             
 1   student_id     6332 non-null   object             
 2   school_id      6332 non-null   object             
 3   enrolled_at    6332 non-null   datetime64[us, UTC]
 4   status         6332 non-null   object             
 5   fee_amount     6332 non-null   Int64              
dtypes: Int64(1), datetime64[us, UTC](1), object(4)
memory usage: 303.1+ KB


In [24]:
print("Duplicate rows:", df_enrollments.duplicated().sum())

print("Duplicate Enrollment IDs:",
      df_enrollments["enrollment_id"].duplicated().sum())

print("\nEnrollment status:")
print(df_enrollments["status"].value_counts())

print("\nFee statistics:")
print(df_enrollments["fee_amount"].describe())

Duplicate rows: 0
Duplicate Enrollment IDs: 0

Enrollment status:
status
Enrolled     5895
Withdrawn     437
Name: count, dtype: int64

Fee statistics:
count           6332.0
mean     147386.037587
std       59355.622559
min            45035.0
25%           96394.25
50%           147303.5
75%           198347.5
max           249992.0
Name: fee_amount, dtype: Float64


In [25]:
df_enrollments_clean = df_enrollments.copy()

print("Shape:", df_enrollments_clean.shape)
print("Duplicates:", df_enrollments_clean.duplicated().sum())

Shape: (6332, 6)
Duplicates: 0


In [26]:
# Lead performance by school

leads_by_school = (
    df_leads_clean
    .groupby("school_id")
    .agg(
        total_leads=("lead_id", "count"),
        qualified_leads=("status", lambda x: (x == "Qualified").sum()),
        converted_leads=("status", lambda x: (x == "Converted").sum())
    )
    .reset_index()
)

leads_by_school.head()

,school_id,total_leads,qualified_leads,converted_leads
0,SCH001,789,158,171
1,SCH002,756,153,158
2,SCH003,732,147,153
3,SCH004,740,127,188
4,SCH005,766,162,176


In [27]:
# Application performance by school

applications_by_school = (
    df_applications_clean
    .groupby("school_id")
    .agg(
        total_applications=("application_id", "count"),
        accepted_applications=("status", lambda x: (x == "Accepted").sum())
    )
    .reset_index()
)

applications_by_school.head()

,school_id,total_applications,accepted_applications
0,SCH001,211,60
1,SCH002,202,66
2,SCH003,184,62
3,SCH004,173,69
4,SCH005,206,62


In [28]:
# Enrollment and revenue performance by school

enrollments_by_school = (
    df_enrollments_clean
    .groupby("school_id")
    .agg(
        total_enrollments=("enrollment_id", "count"),
        enrolled_students=("status", lambda x: (x == "Enrolled").sum()),
        withdrawn_students=("status", lambda x: (x == "Withdrawn").sum()),
        total_revenue=("fee_amount", "sum"),
        average_fee=("fee_amount", "mean")
    )
    .reset_index()
)

enrollments_by_school.head()

,school_id,total_enrollments,enrolled_students,withdrawn_students,total_revenue,average_fee
0,SCH001,60,54,6,9646873,160781.216667
1,SCH002,66,64,2,9720301,147277.287879
2,SCH003,62,60,2,8326621,134300.33871
3,SCH004,69,63,6,11301714,163792.956522
4,SCH005,62,58,4,8986632,144945.677419


In [29]:
school_funnel = (
    df_school_clean
    .merge(leads_by_school, on="school_id", how="left")
    .merge(applications_by_school, on="school_id", how="left")
    .merge(enrollments_by_school, on="school_id", how="left")
)

school_funnel.head()

,school_id,school_name,city,state,region,school_type,boarding_type,total_leads,qualified_leads,converted_leads,total_applications,accepted_applications,total_enrollments,enrolled_students,withdrawn_students,total_revenue,average_fee
0,SCH010,Delhi Public School,Patna,Bihar,East,Private,Day,676,140,155,189,53,53,49,4,8186118,154455.056604
1,SCH043,JG International School,Ahmedabad,Gujarat,West,Private,Day,786,151,171,203,58,58,51,7,8295991,143034.327586
2,SCH045,Navrachana International School,Vadodara,Gujarat,West,Private,Day,733,136,170,193,60,60,53,7,9743204,162386.733333
3,SCH048,Bhavkunj School,Kadi,Gujarat,West,Private,Day,719,136,152,171,54,54,47,7,8440720,156309.62963
4,SCH049,PD Savani Cambridge International School,Surat,Gujarat,West,Private,Day,769,157,184,234,74,74,67,7,10480289,141625.527027


In [30]:
print("Shape:", school_funnel.shape)
print("Missing values:")
print(school_funnel.isna().sum())

Shape: (100, 17)
Missing values:
school_id                0
school_name              0
city                     0
state                    0
region                   0
school_type              0
boarding_type            0
total_leads              0
qualified_leads          0
converted_leads          0
total_applications       0
accepted_applications    0
total_enrollments        0
enrolled_students        0
withdrawn_students       0
total_revenue            0
average_fee              0
dtype: int64


## 2. Business Funnel Analysis

The analysis follows the business funnel:

Leads → Qualified Leads → Applications → Enrollments → Revenue

Key analyses:
- School-level funnel performance
- Regional performance
- Organic vs PPC lead performance
- Campaign efficiency
- Monthly trends
- Revenue and enrollment performance

In [31]:
school_funnel["qualification_rate"] = (
    school_funnel["qualified_leads"] /
    school_funnel["total_leads"] * 100
)

school_funnel["lead_conversion_rate"] = (
    school_funnel["converted_leads"] /
    school_funnel["total_leads"] * 100
)

school_funnel["application_rate"] = (
    school_funnel["total_applications"] /
    school_funnel["total_leads"] * 100
)

school_funnel["enrollment_rate"] = (
    school_funnel["total_enrollments"] /
    school_funnel["total_applications"] * 100
)

school_funnel["withdrawal_rate"] = (
    school_funnel["withdrawn_students"] /
    school_funnel["total_enrollments"] * 100
)

school_funnel["revenue_per_enrollment"] = (
    school_funnel["total_revenue"] /
    school_funnel["total_enrollments"]
)

school_funnel.head()

,school_id,school_name,city,state,region,school_type,boarding_type,total_leads,qualified_leads,converted_leads,...,enrolled_students,withdrawn_students,total_revenue,average_fee,qualification_rate,lead_conversion_rate,application_rate,enrollment_rate,withdrawal_rate,revenue_per_enrollment
0,SCH010,Delhi Public School,Patna,Bihar,East,Private,Day,676,140,155,...,49,4,8186118,154455.056604,20.710059,22.928994,27.958580,28.042328,7.547170,154455.056604
1,SCH043,JG International School,Ahmedabad,Gujarat,West,Private,Day,786,151,171,...,51,7,8295991,143034.327586,19.211196,21.755725,25.826972,28.571429,12.068966,143034.327586
2,SCH045,Navrachana International School,Vadodara,Gujarat,West,Private,Day,733,136,170,...,53,7,9743204,162386.733333,18.553888,23.192360,26.330150,31.088083,11.666667,162386.733333
3,SCH048,Bhavkunj School,Kadi,Gujarat,West,Private,Day,719,136,152,...,47,7,8440720,156309.62963,18.915160,21.140473,23.783032,31.578947,12.962963,156309.62963
4,SCH049,PD Savani Cambridge International School,Surat,Gujarat,West,Private,Day,769,157,184,...,67,7,10480289,141625.527027,20.416125,23.927178,30.429129,31.623932,9.459459,141625.527027


In [32]:
overall_funnel = {
    "Total Schools": school_funnel["school_id"].nunique(),
    "Total Leads": school_funnel["total_leads"].sum(),
    "Qualified Leads": school_funnel["qualified_leads"].sum(),
    "Converted Leads": school_funnel["converted_leads"].sum(),
    "Applications": school_funnel["total_applications"].sum(),
    "Accepted Applications": school_funnel["accepted_applications"].sum(),
    "Enrollments": school_funnel["total_enrollments"].sum(),
    "Enrolled Students": school_funnel["enrolled_students"].sum(),
    "Withdrawals": school_funnel["withdrawn_students"].sum(),
    "Total Revenue": school_funnel["total_revenue"].sum()
}

overall_funnel

{'Total Schools': 100,
 'Total Leads': np.int64(75000),
 'Qualified Leads': np.int64(15062),
 'Converted Leads': np.int64(16337),
 'Applications': np.int64(20000),
 'Accepted Applications': np.int64(6332),
 'Enrollments': np.int64(6332),
 'Enrolled Students': np.int64(5895),
 'Withdrawals': np.int64(437),
 'Total Revenue': np.int64(933248390)}

In [33]:
total_leads = school_funnel["total_leads"].sum()
qualified = school_funnel["qualified_leads"].sum()
converted = school_funnel["converted_leads"].sum()
applications = school_funnel["total_applications"].sum()
enrollments = school_funnel["total_enrollments"].sum()
withdrawals = school_funnel["withdrawn_students"].sum()

overall_rates = {
    "Qualification Rate %": qualified / total_leads * 100,
    "Lead Conversion Rate %": converted / total_leads * 100,
    "Application Rate %": applications / total_leads * 100,
    "Enrollment Rate %": enrollments / applications * 100,
    "Withdrawal Rate %": withdrawals / enrollments * 100,
    "Revenue per Enrollment": school_funnel["total_revenue"].sum() / enrollments
}

overall_rates

{'Qualification Rate %': np.float64(20.082666666666665),
 'Lead Conversion Rate %': np.float64(21.782666666666668),
 'Application Rate %': np.float64(26.666666666666668),
 'Enrollment Rate %': np.float64(31.66),
 'Withdrawal Rate %': np.float64(6.901452937460519),
 'Revenue per Enrollment': np.float64(147386.0375868604)}

In [34]:
top_schools = (
    school_funnel[
        [
            "school_id",
            "school_name",
            "region",
            "total_leads",
            "total_applications",
            "total_enrollments",
            "total_revenue",
            "lead_conversion_rate",
            "enrollment_rate"
        ]
    ]
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

top_schools

,school_id,school_name,region,total_leads,total_applications,total_enrollments,total_revenue,lead_conversion_rate,enrollment_rate
52,SCH091,Bhopal Residential Academy 30,Central,804,220,86,12564182,22.512438,39.090909
39,SCH033,Auckland House School,North,742,233,76,12215655,21.428571,32.618026
9,SCH080,Bengaluru Residential Academy 19,South,788,221,84,11962084,23.984772,38.009050
19,SCH020,Doon Global School,North,783,219,72,11630871,20.945083,32.876712
24,SCH006,Mann School,North,723,195,77,11404113,20.470263,39.487179
96,SCH034,Birla Vidyamandir,North,739,237,79,11373189,24.357240,33.333333
75,SCH004,Hopetown Girls' School,North,740,173,69,11301714,25.405405,39.884393
10,SCH100,Bengaluru Residential Academy 39,South,757,229,73,11205731,22.324967,31.877729
40,SCH067,Shimla Residential Academy 06,North,775,212,75,11131889,20.903226,35.377358
57,SCH055,Indus International School,West,738,209,77,10986466,21.815718,36.842105


In [35]:
high_volume_low_conversion = (
    school_funnel[
        [
            "school_id",
            "school_name",
            "region",
            "total_leads",
            "qualified_leads",
            "converted_leads",
            "total_applications",
            "total_enrollments",
            "lead_conversion_rate",
            "enrollment_rate",
            "total_revenue"
        ]
    ]
    .sort_values(
        ["total_leads", "lead_conversion_rate"],
        ascending=[False, True]
    )
)

high_volume_low_conversion.head(10)

,school_id,school_name,region,total_leads,qualified_leads,converted_leads,total_applications,total_enrollments,lead_conversion_rate,enrollment_rate,total_revenue
86,SCH021,The Aryan School,North,811,173,167,218,70,20.591862,32.110092,10548431
20,SCH092,Dehradun Residential Academy 31,North,809,157,171,210,51,21.137206,24.285714,7488273
52,SCH091,Bhopal Residential Academy 30,Central,804,170,181,220,86,22.512438,39.090909,12564182
77,SCH008,Mussoorie International School,North,794,158,166,206,62,20.906801,30.097087,9194678
85,SCH018,Kasiga School,North,793,145,180,214,61,22.698613,28.504673,9369926
42,SCH069,Ranchi Residential Academy 08,East,791,156,179,206,68,22.629583,33.009709,9720024
44,SCH001,Jain International School,South,789,158,171,211,60,21.673004,28.436019,9646873
9,SCH080,Bengaluru Residential Academy 19,South,788,169,189,221,84,23.984772,38.009050,11962084
1,SCH043,JG International School,West,786,151,171,203,58,21.755725,28.571429,8295991
50,SCH081,Bhopal Residential Academy 20,Central,786,162,177,213,70,22.519084,32.863850,9913779


In [36]:
region_performance = (
    school_funnel
    .groupby("region")
    .agg(
        schools=("school_id", "nunique"),
        total_leads=("total_leads", "sum"),
        qualified_leads=("qualified_leads", "sum"),
        converted_leads=("converted_leads", "sum"),
        applications=("total_applications", "sum"),
        enrollments=("total_enrollments", "sum"),
        withdrawals=("withdrawn_students", "sum"),
        revenue=("total_revenue", "sum")
    )
    .reset_index()
)

region_performance["lead_conversion_rate"] = (
    region_performance["converted_leads"]
    / region_performance["total_leads"] * 100
)

region_performance["enrollment_rate"] = (
    region_performance["enrollments"]
    / region_performance["applications"] * 100
)

region_performance["withdrawal_rate"] = (
    region_performance["withdrawals"]
    / region_performance["enrollments"] * 100
)

region_performance["revenue_per_enrollment"] = (
    region_performance["revenue"]
    / region_performance["enrollments"]
)

region_performance.sort_values("revenue", ascending=False)

,region,schools,total_leads,qualified_leads,converted_leads,applications,enrollments,withdrawals,revenue,lead_conversion_rate,enrollment_rate,withdrawal_rate,revenue_per_enrollment
2,North,47,35230,7037,7638,9298,2904,185,430179071,21.680386,31.232523,6.370523,148133.288912
4,West,27,20155,4021,4461,5449,1753,126,257340388,22.133466,32.171041,7.187678,146799.993155
1,East,9,6651,1316,1439,1784,563,43,83813660,21.635844,31.558296,7.637655,148869.73357
3,South,9,6831,1411,1473,1836,571,43,83684282,21.563461,31.100218,7.530648,146557.411559
0,Central,8,6133,1277,1326,1633,541,40,78230989,21.620740,33.129210,7.393715,144604.415896


In [37]:
source_performance = (
    df_leads_clean
    .groupby("lead_source")
    .agg(
        total_leads=("lead_id", "count"),
        qualified_leads=("status", lambda x: (x == "Qualified").sum()),
        converted_leads=("status", lambda x: (x == "Converted").sum())
    )
    .reset_index()
)

source_performance["qualification_rate"] = (
    source_performance["qualified_leads"]
    / source_performance["total_leads"] * 100
)

source_performance["conversion_rate"] = (
    source_performance["converted_leads"]
    / source_performance["total_leads"] * 100
)

source_performance.sort_values("conversion_rate", ascending=False)

,lead_source,total_leads,qualified_leads,converted_leads,qualification_rate,conversion_rate
0,Organic,43469,8788,9475,20.216706,21.797143
1,PPC,31531,6274,6862,19.897878,21.762710


In [38]:
campaign_performance = (
    df_leads_clean
    .dropna(subset=["campaign_id"])
    .groupby("campaign_id")
    .agg(
        total_leads=("lead_id", "count"),
        qualified_leads=("status", lambda x: (x == "Qualified").sum()),
        converted_leads=("status", lambda x: (x == "Converted").sum())
    )
    .reset_index()
)

campaign_performance["qualification_rate"] = (
    campaign_performance["qualified_leads"]
    / campaign_performance["total_leads"] * 100
)

campaign_performance["conversion_rate"] = (
    campaign_performance["converted_leads"]
    / campaign_performance["total_leads"] * 100
)

campaign_performance.sort_values(
    "conversion_rate",
    ascending=False
).head(10)

,campaign_id,total_leads,qualified_leads,converted_leads,qualification_rate,conversion_rate
204,CMP0205,99,13,33,13.131313,33.333333
7,CMP0008,83,15,27,18.072289,32.530120
35,CMP0036,96,21,31,21.875000,32.291667
93,CMP0094,109,17,35,15.596330,32.110092
38,CMP0039,114,17,36,14.912281,31.578947
180,CMP0181,119,25,37,21.008403,31.092437
165,CMP0166,102,18,31,17.647059,30.392157
227,CMP0228,111,18,33,16.216216,29.729730
36,CMP0037,106,16,31,15.094340,29.245283
232,CMP0233,114,22,33,19.298246,28.947368


In [41]:
[c for c in globals() if "campaign" in c.lower()]

['campaign_performance']

In [42]:
[c for c in globals() if c.startswith("df_")]

['df_school',
 'df_school_clean',
 'df_leads',
 'df_leads_clean',
 'df_applications',
 'df_applications_clean',
 'df_enrollments',
 'df_enrollments_clean']

In [43]:
[c for c in globals() if any(x in c.lower() for x in ["client", "bq", "bigquery", "project"])]

['bigquery', 'PROJECT_ID', 'client']

In [45]:
print("PROJECT_ID:", PROJECT_ID)

print("\nDatasets:")
for dataset in client.list_datasets():
    print(dataset.dataset_id)

PROJECT_ID: project-53983553-3770-4fa0-a04

Datasets:
allschoolscolleges


In [46]:
print("PROJECT_ID:", PROJECT_ID)

for dataset in client.list_datasets():
    print("DATASET:", dataset.dataset_id)

PROJECT_ID: project-53983553-3770-4fa0-a04
DATASET: allschoolscolleges


In [47]:
tables = client.list_tables(f"{PROJECT_ID}.allschoolscolleges")

for table in tables:
    print(table.table_id)

dim_school
fact_applications
fact_enrollments
fact_leads
fact_ppc


In [48]:
query_campaigns = f"""
SELECT *
FROM `{PROJECT_ID}.allschoolscolleges.fact_ppc`
"""

df_campaigns = client.query(query_campaigns).to_dataframe()

print("Shape:", df_campaigns.shape)
print(df_campaigns.columns.tolist())
df_campaigns.head()

Shape: (6642, 9)
['school_id', 'month', 'leads', 'impressions', 'clicks', 'spend', 'applications', 'enrollments', 'campaign_id']


,school_id,month,leads,impressions,clicks,spend,applications,enrollments,campaign_id
0,SCH002,2025-12-01 00:00:00+00:00,1,66,2,75.51,0,0,CMP0126
1,SCH006,2024-05-01 00:00:00+00:00,1,50,2,44.29,0,0,CMP0074
2,SCH010,2023-05-01 00:00:00+00:00,1,27,2,53.07,0,0,CMP0028
3,SCH011,2021-06-01 00:00:00+00:00,1,54,2,38.99,0,0,CMP0071
4,SCH014,2021-02-01 00:00:00+00:00,1,41,2,26.26,0,0,CMP0266


In [49]:
campaign_roi = (
    df_campaigns
    .groupby("campaign_id")
    .agg(
        total_leads=("leads", "sum"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
        total_spend=("spend", "sum"),
        total_applications=("applications", "sum"),
        total_enrollments=("enrollments", "sum")
    )
    .reset_index()
)

# Avoid division-by-zero
campaign_roi["ctr"] = (
    campaign_roi["total_clicks"]
    / campaign_roi["total_impressions"].replace(0, float("nan"))
) * 100

campaign_roi["cost_per_lead"] = (
    campaign_roi["total_spend"]
    / campaign_roi["total_leads"].replace(0, float("nan"))
)

campaign_roi["cost_per_application"] = (
    campaign_roi["total_spend"]
    / campaign_roi["total_applications"].replace(0, float("nan"))
)

campaign_roi["application_rate"] = (
    campaign_roi["total_applications"]
    / campaign_roi["total_leads"].replace(0, float("nan"))
) * 100

campaign_roi["enrollment_rate"] = (
    campaign_roi["total_enrollments"]
    / campaign_roi["total_applications"].replace(0, float("nan"))
) * 100

campaign_roi.sort_values(
    "cost_per_lead",
    ascending=True
).head(10)


,campaign_id,total_leads,total_impressions,total_clicks,total_spend,total_applications,total_enrollments,ctr,cost_per_lead,cost_per_application,application_rate,enrollment_rate
235,CMP0236,99,4987,317,7212.98,24,2,6.356527,72.858384,300.540833,24.242424,8.333333
45,CMP0046,108,5549,438,9016.43,26,1,7.893314,83.485463,346.785769,24.074074,3.846154
86,CMP0087,111,5226,398,9350.00,19,1,7.615767,84.234234,492.105263,17.117117,5.263158
91,CMP0092,107,6101,462,9136.70,26,1,7.572529,85.38972,351.411538,24.299065,3.846154
84,CMP0085,104,4756,366,9065.77,21,0,7.695542,87.170865,431.703333,20.192308,0.0
250,CMP0251,109,4550,404,9523.40,24,2,8.879121,87.370642,396.808333,22.018349,8.333333
117,CMP0118,108,5275,424,9492.86,21,1,8.037915,87.896852,452.040952,19.444444,4.761905
155,CMP0156,95,5351,446,8457.24,21,2,8.334891,89.023579,402.725714,22.105263,9.52381
212,CMP0213,100,5707,365,8924.80,17,0,6.395654,89.248,524.988235,17.0,0.0
200,CMP0201,111,5628,398,10053.82,21,3,7.071784,90.574955,478.753333,18.918919,14.285714


In [50]:
monthly_ppc = (
    df_campaigns
    .groupby("month")
    .agg(
        leads=("leads", "sum"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spend=("spend", "sum"),
        applications=("applications", "sum"),
        enrollments=("enrollments", "sum")
    )
    .reset_index()
)

monthly_ppc["ctr"] = (
    monthly_ppc["clicks"]
    / monthly_ppc["impressions"].replace(0, float("nan"))
) * 100

monthly_ppc["cost_per_lead"] = (
    monthly_ppc["spend"]
    / monthly_ppc["leads"].replace(0, float("nan"))
)

monthly_ppc["application_rate"] = (
    monthly_ppc["applications"]
    / monthly_ppc["leads"].replace(0, float("nan"))
) * 100

monthly_ppc["enrollment_rate"] = (
    monthly_ppc["enrollments"]
    / monthly_ppc["applications"].replace(0, float("nan"))
) * 100

monthly_ppc.sort_values("month")

,month,leads,impressions,clicks,spend,applications,enrollments,ctr,cost_per_lead,application_rate,enrollment_rate
0,2021-01-01 00:00:00+00:00,461,23045,1982,55130.65,75,2,8.600564,119.589262,16.26898,2.666667
1,2021-02-01 00:00:00+00:00,427,21731,1877,51638.36,82,3,8.63743,120.932927,19.203747,3.658537
2,2021-03-01 00:00:00+00:00,469,23326,2101,60510.57,83,3,9.007117,129.020405,17.697228,3.614458
3,2021-04-01 00:00:00+00:00,466,23237,2022,51875.12,92,8,8.70164,111.32,19.742489,8.695652
4,2021-05-01 00:00:00+00:00,472,22872,2206,55440.76,87,1,9.644981,117.459237,18.432203,1.149425
...,...,...,...,...,...,...,...,...,...,...,...
62,2026-03-01 00:00:00+00:00,467,23852,2027,54271.64,87,2,8.498239,116.213362,18.62955,2.298851
63,2026-04-01 00:00:00+00:00,430,21375,1916,52238.28,80,4,8.963743,121.484372,18.604651,5.0
64,2026-05-01 00:00:00+00:00,501,22340,2347,64536.61,103,5,10.505819,128.815589,20.558882,4.854369
65,2026-06-01 00:00:00+00:00,495,24361,2098,57674.62,96,6,8.612126,116.514384,19.393939,6.25


In [51]:
monthly_leads = (
    df_leads_clean
    .assign(month=df_leads_clean["lead_date"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(
        total_leads=("lead_id", "count"),
        qualified_leads=("status", lambda x: (x == "Qualified").sum()),
        converted_leads=("status", lambda x: (x == "Converted").sum())
    )
    .reset_index()
)

monthly_leads["qualification_rate"] = (
    monthly_leads["qualified_leads"]
    / monthly_leads["total_leads"].replace(0, float("nan"))
) * 100

monthly_leads["conversion_rate"] = (
    monthly_leads["converted_leads"]
    / monthly_leads["total_leads"].replace(0, float("nan"))
) * 100

monthly_leads.sort_values("month")

/tmp/ipykernel_1052/3015688911.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .assign(month=df_leads_clean["lead_date"].dt.to_period("M").astype(str))


,month,total_leads,qualified_leads,converted_leads,qualification_rate,conversion_rate
0,2021-01,1102,221,224,20.054446,20.326679
1,2021-02,991,216,202,21.796165,20.383451
2,2021-03,1160,233,282,20.086207,24.310345
3,2021-04,1123,224,254,19.946572,22.617988
4,2021-05,1132,243,255,21.466431,22.526502
...,...,...,...,...,...,...
62,2026-03,1108,217,231,19.584838,20.848375
63,2026-04,1091,225,242,20.623281,22.181485
64,2026-05,1223,224,288,18.315617,23.548651
65,2026-06,1163,254,259,21.840069,22.269991


In [54]:
monthly_leads["month"] = pd.to_datetime(
    monthly_leads["month"]
).dt.to_period("M").astype(str)

monthly_ppc["month"] = pd.to_datetime(
    monthly_ppc["month"]
).dt.to_period("M").astype(str)

/tmp/ipykernel_1052/2105897300.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ).dt.to_period("M").astype(str)


In [55]:
monthly_combined = (
    monthly_leads
    .merge(
        monthly_ppc,
        on="month",
        how="outer",
        suffixes=("_leads", "_ppc")
    )
    .sort_values("month")
)

monthly_combined.head(10)

,month,total_leads,qualified_leads,converted_leads,qualification_rate,conversion_rate,leads,impressions,clicks,spend,applications,enrollments,ctr,cost_per_lead,application_rate,enrollment_rate
0,2021-01,1102,221,224,20.054446,20.326679,461,23045,1982,55130.65,75,2,8.600564,119.589262,16.26898,2.666667
1,2021-02,991,216,202,21.796165,20.383451,427,21731,1877,51638.36,82,3,8.63743,120.932927,19.203747,3.658537
2,2021-03,1160,233,282,20.086207,24.310345,469,23326,2101,60510.57,83,3,9.007117,129.020405,17.697228,3.614458
3,2021-04,1123,224,254,19.946572,22.617988,466,23237,2022,51875.12,92,8,8.70164,111.32,19.742489,8.695652
4,2021-05,1132,243,255,21.466431,22.526502,472,22872,2206,55440.76,87,1,9.644981,117.459237,18.432203,1.149425
5,2021-06,1113,221,235,19.856244,21.114106,465,22885,2043,53286.57,90,5,8.927245,114.594774,19.354839,5.555556
6,2021-07,1182,258,264,21.827411,22.335025,482,24301,2201,58493.81,98,7,9.05724,121.356452,20.33195,7.142857
7,2021-08,1156,198,268,17.128028,23.183391,480,23877,2005,51633.31,92,5,8.397202,107.569396,19.166667,5.434783
8,2021-09,1092,235,249,21.520147,22.802198,455,22549,1932,49371.14,90,4,8.568007,108.508,19.78022,4.444444
9,2021-10,1093,207,222,18.938701,20.311070,449,22194,2059,59866.61,95,7,9.277282,133.333207,21.158129,7.368421


In [56]:
print("Shape:", monthly_combined.shape)
print("\nMissing values:")
print(monthly_combined.isna().sum())

Shape: (67, 16)

Missing values:
month                 0
total_leads           0
qualified_leads       0
converted_leads       0
qualification_rate    0
conversion_rate       0
leads                 0
impressions           0
clicks                0
spend                 0
applications          0
enrollments           0
ctr                   0
cost_per_lead         0
application_rate      0
enrollment_rate       0
dtype: int64


In [57]:
print("=== SCHOOL FUNNEL ===")
print("Shape:", school_funnel.shape)
print("Duplicates:", school_funnel.duplicated().sum())
print("Missing values:", school_funnel.isna().sum().sum())

print("\n=== MONTHLY COMBINED ===")
print("Shape:", monthly_combined.shape)
print("Missing values:", monthly_combined.isna().sum().sum())

print("\n=== CAMPAIGN ROI ===")
print("Shape:", campaign_roi.shape)
print("Missing values:", campaign_roi.isna().sum().sum())

=== SCHOOL FUNNEL ===
Shape: (100, 23)
Duplicates: 0
Missing values: 0

=== MONTHLY COMBINED ===
Shape: (67, 16)
Missing values: 0

=== CAMPAIGN ROI ===
Shape: (300, 12)
Missing values: 0


In [58]:
print("=== DATASETS AVAILABLE ===")

for name in [
    "df_school_clean",
    "df_leads_clean",
    "df_applications_clean",
    "df_enrollments_clean",
    "df_campaigns",
    "school_funnel",
    "region_performance",
    "source_performance",
    "campaign_roi",
    "monthly_ppc",
    "monthly_leads",
    "monthly_combined"
]:
    if name in globals():
        print("✓", name)

=== DATASETS AVAILABLE ===
✓ df_school_clean
✓ df_leads_clean
✓ df_applications_clean
✓ df_enrollments_clean
✓ df_campaigns
✓ school_funnel
✓ region_performance
✓ source_performance
✓ campaign_roi
✓ monthly_ppc
✓ monthly_leads
✓ monthly_combined


## 3. Key Business Insights

Insights will be documented after completing the analysis and validating
the results against the BigQuery source data.